# l02 · 1일차 2교시 · requests로 날씨 요청
**목표**: 파이썬 코드로 날씨 API에 요청을 보내고, 응답에서 현재 기온을 꺼냅니다.

요청할 때마다 밟는 **4단계**를 익히는 시간입니다.

| 단계 | 하는 일 | 코드 |
|---|---|---|
| ① 주소 만들기 | 어느 서버에 무엇을 달라고 할지 | `url`, `params` 딕셔너리 |
| ② 보내기 | 그 주소로 요청을 보냄 | `requests.get()` |
| ③ 확인하기 | 성공했는지 결과 번호를 봄 | `res.status_code` |
| ④ 꺼내기 | JSON을 딕셔너리로 바꿔 값을 꺼냄 | `res.json()["키"]` |

> **노트북 사용법** · 셀을 클릭하고 **Shift+Enter**를 누르면 실행되고 다음 셀로 넘어갑니다.
> 셀 왼쪽의 `[1] [2]…`는 **실행한 순서**입니다. 위에서부터 차례로 실행하세요. 중간 셀을 건너뛰면 `NameError`가 납니다.
> 처음부터 깨끗하게 다시 하려면 위쪽 메뉴의 **Restart → Run All**을 누릅니다.

## 0. 준비 · requests 설치
`requests`는 파이썬에 기본으로 들어 있지 않은 **외부 라이브러리**입니다. 처음 한 번만 설치합니다.
노트북 오른쪽 위 **커널**이 수업용 가상환경(.venv 또는 ge_env)인지 먼저 확인하세요. `%pip`는 지금 고른 커널에 설치합니다.

In [ ]:
%pip install requests

## 1. requests는 파이썬용 브라우저
| | 브라우저 | requests |
|---|---|---|
| 요청 보내기 | 주소창에 주소 입력 | `requests.get(주소)` |
| 받은 응답 | 화면으로 그려 보여 줌 | 변수에 담아 코드로 다룸 |
| 쓰는 쪽 | 사람 | 파이썬 코드 |

## 2. 첫 요청 · ② 보내기
l01에서 브라우저로 열었던 바로 그 주소로 요청을 보냅니다.

In [1]:
import requests   # 설치한 라이브러리를 불러옴

url = ("https://api.open-meteo.com/v1/forecast"
       "?latitude=37.5665&longitude=126.978"
       "&current=temperature_2m")   # 괄호 안에서 문자열을 여러 줄로 나눠 써도 한 줄로 이어짐

res = requests.get(url)   # 요청을 보내고, 돌아온 응답을 res 변수에 담음
print(res)                # <Response [200]> → res는 '응답 객체', 200은 성공 번호

<Response [200]>


## 3. 응답 객체 들여다보기
`res`는 응답의 여러 정보를 담은 **객체**입니다. 점(`.`)을 찍어 꺼내 봅니다.

| 쓰는 법 | 보여 주는 것 |
|---|---|
| `res.status_code` | 결과 번호 (200이면 성공) |
| `res.url` | 실제로 보낸 주소 |
| `res.headers["Content-Type"]` | 응답 형식 (JSON인지 HTML인지) |
| `res.text` | 본문을 **글자(str)** 그대로 |
| `res.json()` | 본문을 **딕셔너리(dict)**로 바꿔서 |

In [2]:
print(res.status_code)
print(res.url)
print(res.headers["Content-Type"])   # application/json → JSON으로 왔다는 뜻
print(res.text[:60])                 # 글자가 길어서 앞 60글자만 보기

200
https://api.open-meteo.com/v1/forecast?latitude=37.5665&longitude=126.978&current=temperature_2m
application/json; charset=utf-8
{"latitude":37.55,"longitude":127.0,"generationtime_ms":0.22


## 4. `.text`는 글자, `.json()`은 딕셔너리
`res.text`는 글자(str)라서 `["current"]`로 꺼낼 수 없습니다.
`.json()`으로 딕셔너리로 바꾼 뒤에야 값을 꺼낼 수 있습니다. (l01의 `json.loads(res.text)`와 같은 일)

- 눈으로 확인할 때는 `.text`
- 값을 꺼낼 때는 `.json()` ← **괄호를 꼭 붙입니다**

In [3]:
print(type(res.text))     # <class 'str'>
print(type(res.json()))   # <class 'dict'>

data = res.json()                         # ④ 꺼내기: 딕셔너리로 바꿈
print(data["current"]["temperature_2m"])  # 현재 기온

<class 'str'>
<class 'dict'>
11.9


아래 두 셀은 **일부러 오류를 내는 예시**입니다. 어떤 오류 메시지가 나오는지 읽어 보세요.

In [4]:
res.text["current"]   # 글자에서 꺼내려고 함 → TypeError: string indices must be integers

TypeError: string indices must be integers, not 'str'

In [5]:
print(res.json)   # 괄호를 빠뜨림 → <bound method ...>가 출력됨 (값이 아니라 '메서드' 자체)

<bound method Response.json of <Response [200]>>


## 5. 상태 코드 · ③ 확인하기
서버는 응답마다 **세 자리 번호**를 붙여 보냅니다. 식당으로 치면 "요리 나왔어요(200)", "그런 메뉴 없어요(404)"입니다.

| 코드 | 뜻 | 할 일 |
|---|---|---|
| **200** | 성공, 데이터가 있음 | `.json()`으로 꺼내기 |
| **404** | 그런 주소·문서가 없음 | 주소 확인 |
| 4xx | 내 요청 문제 | 주소·파라미터 고치기 |
| 5xx | 서버 문제 | 잠시 뒤 다시 시도 |
| 401 | 키가 없거나 틀림 (2일차) | 키 확인 |
| 429 | 너무 자주 요청함 (2일차) | 요청 사이에 쉬기 |

**200일 때만 ④ 꺼내기로 넘어갑니다.** 실패한 응답에서 값을 꺼내려다 프로그램이 멈추는 일을 막습니다.

In [6]:
# 일부러 없는 경로로 요청해 봅니다
bad = requests.get("https://api.open-meteo.com/v1/없는경로")
print(bad.status_code)   # 404

if bad.status_code == 200:
    print("성공! 값을 꺼냅니다")
else:
    print("요청 실패:", bad.status_code)

404
요청 실패: 404


## 6. params · ① 주소 만들기를 편하게
주소에 `?`와 `&`를 손으로 이어 붙이면 빠뜨리기 쉽습니다.
`params`에 **딕셔너리**로 적어 넘기면 requests가 `?`, `&`를 알아서 붙여 주고, 특수 문자(`/` → `%2F`)도 바꿔 줍니다.

- 키: API 문서에 적힌 **파라미터 이름** (따옴표 필수)
- 값: 숫자·문자열 그대로

In [7]:
url = "https://api.open-meteo.com/v1/forecast"   # ? 앞까지만 적음
params = {
    "latitude": 37.5665,           # 위도 (서울)
    "longitude": 126.978,          # 경도 (서울)
    "current": "temperature_2m",   # 현재 기온
    "hourly": "temperature_2m",    # 시간별 기온
    "timezone": "Asia/Seoul",      # 한국 시간으로
    "forecast_days": 1,            # 오늘 하루치만
}
res = requests.get(url, params=params)   # params= 로 딕셔너리를 넘김
print(res.url)   # requests가 완성한 실제 주소. ?와 &가 붙어 있는지 확인

https://api.open-meteo.com/v1/forecast?latitude=37.5665&longitude=126.978&current=temperature_2m&hourly=temperature_2m&timezone=Asia%2FSeoul&forecast_days=1


> **자주 하는 실수** · `{latitude: 37.5665}`처럼 키에 따옴표를 빼면 `NameError`가 납니다. `{"latitude": 37.5665}`로 씁니다.

## 7. 중첩 꺼내기 · ④ 꺼내기
응답의 뼈대는 이렇게 생겼습니다.
```
{
  "current": {"temperature_2m": 14.3},
  "hourly": {
      "time":           ["…T00:00", "…T01:00", …],
      "temperature_2m": [12.1,      11.6,      …]
  }
}
```
`time`과 `temperature_2m`은 **같은 번호끼리 짝**입니다. `time[0]` 시각의 기온이 `temperature_2m[0]`입니다.

In [8]:
data = res.json()
print(data["current"]["temperature_2m"])   # 현재 기온

hourly = data["hourly"]          # 시간별 데이터 부분만 따로 담아 두면 짧게 쓸 수 있음
for i in range(3):               # i = 0, 1, 2
    print(hourly["time"][i], hourly["temperature_2m"][i])

11.9
2026-10-21T00:00 14.5
2026-10-21T01:00 14.0
2026-10-21T02:00 13.4


## 8. [직접 해 보기] 다른 도시 기온 · 5분
위도·경도만 바꾸면 다른 도시 기온이 나옵니다. 출력 글자의 도시 이름도 함께 바꾸세요.

| 도시 | 위도 | 경도 |
|---|---|---|
| 서울 | 37.5665 | 126.9780 |
| 부산 | 35.1796 | 129.0756 |
| 제주 | 33.4996 | 126.5312 |
| 대전 | 36.3504 | 127.3845 |

In [9]:
url = "https://api.open-meteo.com/v1/forecast"
params = {"latitude": 35.1796,      # ← 여기를 바꿔 보세요
          "longitude": 129.0756,    # ← 여기도
          "current": "temperature_2m",
          "timezone": "Asia/Seoul"}
res = requests.get(url, params=params)
data = res.json()
temp = data["current"]["temperature_2m"]
print("부산 현재 기온:", temp, "°C")

부산 현재 기온: 18.0 °C


## 9. 요청 4단계 정리
지금까지 배운 것을 4단계로 나눠 한 셀에 모았습니다. **3교시 위키백과도, 2일차 영화·공연도 이 모양 그대로**입니다.

In [10]:
import requests

# ① 주소 만들기
url = "https://api.open-meteo.com/v1/forecast"
params = {"latitude": 37.5665, "longitude": 126.978,
          "current": "temperature_2m", "timezone": "Asia/Seoul"}

# ② 보내기
res = requests.get(url, params=params)

# ③ 확인하기
if res.status_code == 200:
    # ④ 꺼내기
    data = res.json()
    print("서울 현재 기온:", data["current"]["temperature_2m"])
else:
    print("요청 실패:", res.status_code)

서울 현재 기온: 11.9


## 2교시 확인 퀴즈
**Q1.** `print(res)` 결과가 `<Response [200]>`일 때 현재 기온을 꺼내는 줄은?
① `res["current"]` ② `res.text["current"]` ③ `res.json()["current"]`

**Q2.** 무엇이 출력될까요?
```python
code = 404
if code == 200:
    print("성공")
elif code < 500:
    print("요청 확인")
else:
    print("서버 확인")
```

**Q3.** 무엇이 출력될까요?
```python
data = {"hourly": {"time": ["00:00", "01:00"], "temp": [12.1, 11.6]}}
h = data["hourly"]
print(h["temp"][1])
```

<details><summary>정답 보기</summary>

Q1 ③ · Q2 `요청 확인` · Q3 `11.6`
</details>

➡️ 다음 파일: **l03_3교시_위키백과.ipynb**